# Train YOLO on Google Colab

Use this notebook to train a custom YOLO model on your labeled dataset.

## What you need to edit
1. `DATASET_ROOT`
2. `CLASS_NAMES`

Expected YOLO dataset structure:

```text
DATASET_ROOT/
  images/
    train/   -> .jpg/.jpeg/.png
    val/     -> .jpg/.jpeg/.png
  labels/
    train/   -> .txt (same basename as image)
    val/     -> .txt (same basename as image)
```


In [ ]:
# 1) Config (edit this)
DATASET_ROOT = "/content/drive/MyDrive/room_sign_dataset"
CLASS_NAMES = ["230", "232", "226", "224"]

BASE_MODEL = "yolo11n.pt"   # e.g. yolo11n.pt, yolo11s.pt
EPOCHS = 50
IMG_SIZE = 640
BATCH_SIZE = 16
RUN_NAME = "yolo_custom_colab"
PROJECT_DIR = "/content/runs/detect"

MOUNT_DRIVE = True
EXPORT_TO_DRIVE_DIR = "/content/drive/MyDrive/yolo-trained-models"

In [ ]:
# 2) Install dependencies + optional Drive mount
import subprocess, sys

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics>=8.3.0', 'pyyaml'])
print('Dependencies installed.')

In [ ]:
# 3) Validate dataset + generate data.yaml
from pathlib import Path
import yaml

dataset_root = Path(DATASET_ROOT)
if not dataset_root.exists():
    raise FileNotFoundError(f'DATASET_ROOT does not exist: {dataset_root}')

train_images_dir = dataset_root / 'images' / 'train'
val_images_dir = dataset_root / 'images' / 'val'
train_labels_dir = dataset_root / 'labels' / 'train'
val_labels_dir = dataset_root / 'labels' / 'val'

for d in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
    if not d.exists() or not d.is_dir():
        raise FileNotFoundError(f'Missing required directory: {d}')

image_exts = {'.jpg', '.jpeg', '.png'}
train_images = sorted([p for p in train_images_dir.glob('*') if p.suffix.lower() in image_exts and p.is_file()])
val_images = sorted([p for p in val_images_dir.glob('*') if p.suffix.lower() in image_exts and p.is_file()])

if not train_images:
    raise ValueError(f'No training images found in: {train_images_dir}')
if not val_images:
    raise ValueError(f'No validation images found in: {val_images_dir}')

train_label_stems = {p.stem for p in train_labels_dir.glob('*.txt') if p.is_file()}
val_label_stems = {p.stem for p in val_labels_dir.glob('*.txt') if p.is_file()}

missing_train = [img.name for img in train_images if img.stem not in train_label_stems]
missing_val = [img.name for img in val_images if img.stem not in val_label_stems]
if missing_train:
    raise ValueError('Missing train labels for images, e.g. ' + ', '.join(missing_train[:10]))
if missing_val:
    raise ValueError('Missing val labels for images, e.g. ' + ', '.join(missing_val[:10]))

if not CLASS_NAMES:
    raise ValueError('CLASS_NAMES cannot be empty.')

data_yaml = {
    'path': str(dataset_root),
    'train': 'images/train',
    'val': 'images/val',
    'names': {i: name for i, name in enumerate(CLASS_NAMES)},
}

data_yaml_path = Path('/content/yolo_data.yaml')
with data_yaml_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False, allow_unicode=False)

print(f'Train images: {len(train_images)}')
print(f'Val images:   {len(val_images)}')
print(f'Classes:      {CLASS_NAMES}')
print(f'Generated:    {data_yaml_path}')

In [ ]:
# 4) Train
from ultralytics import YOLO

model = YOLO(BASE_MODEL)
results = model.train(
    data=str(data_yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
)

best_model_path = Path(results.save_dir) / 'weights' / 'best.pt'
if not best_model_path.exists():
    raise FileNotFoundError(f'Training finished but best.pt not found: {best_model_path}')

print('Training complete.')
print('Best model:', best_model_path)

In [ ]:
# 5) Copy trained model to Drive
import shutil

export_dir = Path(EXPORT_TO_DRIVE_DIR)
export_dir.mkdir(parents=True, exist_ok=True)
target = export_dir / f'{RUN_NAME}_best.pt'
shutil.copy2(best_model_path, target)
print('Copied model to:', target)